# Day 3 · 1교시 [실습 보조] 에이전트 기반 크롤러의 아래층 — `01_crawler`

## 실습 목표

교안 1교시에서 에이전트에게 "이 페이지 긁어줘"라고 시키면 **한 방에 만들어진다.**
이 노트북은 그 **아래층**을 직접 겪는다 — 요청→파싱→정제→Item 까지 파이썬으로.
아래층을 알고 나면 에이전트가 만든 코드가 블랙박스가 아니라 **대조해서 검증할 수 있는 것**이 된다.

| 순서 | 내용 | 교안 연결 |
|------|------|----------|
| 1 | 요청 (httpx + 예의: User-Agent·대기) | 1.2 · 1.3 |
| 2 | 파싱 (F12로 확인한 구조 그대로) | 1.4 |
| 3 | 정제 (중복 제거·정규화 → Item) | 1.6 · Day2 ERD |
| 4 | 동적 페이지 (화면 말고 데이터) | 1.7 |
| 5 | (선택) LLM 한 줄 요약 | 2교시 예고 |

대상은 크롤링 연습 전용 사이트 **`books.toscrape.com`**(책 1,000권 · 50페이지).


## 1. 요청 — 예의를 코드에 넣는다

`User-Agent`로 자신을 밝히고, 타임아웃을 걸고, 요청 사이에 쉰다(교안 1.2).
`robots.txt`도 코드로 확인하는 습관 — 있으면 규칙을 따르고, 없으면(404) 넘어간다.

In [1]:
import time, httpx
from bs4 import BeautifulSoup

BASE = "https://books.toscrape.com/"
# 예의(1.2): 누가 왜 긁는지 밝히는 User-Agent(HTTP 헤더는 ASCII), 요청 사이 대기
HEADERS = {"User-Agent": "vibe-coding-course-crawler/1.0 (educational practice)"}

r = httpx.get(BASE, headers=HEADERS, timeout=10)
r.encoding = "utf-8"                 # 가격의 £ 가 깨지지 않게 (교안 1.3)
print("요청 OK:", r.status_code, "|", len(r.text), "bytes")

rb = httpx.get(BASE + "robots.txt", headers=HEADERS, timeout=10)
print("robots.txt:", rb.status_code, "(404 = 선언된 크롤 규칙 없음)")
time.sleep(0.5)

요청 OK: 200 | 51274 bytes
robots.txt: 404 (404 = 선언된 크롤 규칙 없음)


## 2. 파싱 — F12로 확인한 구조 그대로

교안 1.4에서 개발자 도구로 직접 확인한 그 구조를 쓴다. 한 권의 블록은 `article.product_pod` 이고,
**여기 함정이 두 개** 있다:

- 제목: `h3 a` 의 **본문 텍스트는 `A Light in the ...` 로 잘려 있다.** 온전한 제목은 `title` **속성**에 있다.
- 별점: 별 아이콘(`<i>`)은 **항상 5개**다. 진짜 별점은 `class="star-rating Three"` 의 **클래스 이름**에 있다.

In [2]:
RATING_WORDS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def parse_books(html, base=BASE):
    """HTML → [{title, price, rating, url}] (교안 1.4에서 확인한 구조)."""
    from urllib.parse import urljoin
    soup = BeautifulSoup(html, "html.parser")   # lxml 없이 표준 파서
    out = []
    for pod in soup.select("article.product_pod"):
        link = pod.select_one("h3 a")
        cls = pod.select_one("p.star-rating").get("class", [])
        out.append({
            "title":  link["title"],                                  # 본문 아닌 '속성'
            "price":  pod.select_one("p.price_color").get_text(strip=True),
            "rating": next((RATING_WORDS[c] for c in cls if c in RATING_WORDS), None),
            "url":    urljoin(base, link["href"]),                    # 상대→절대
        })
    return out

books = parse_books(r.text)
print("추출 건수:", len(books))
print("첫 권:", books[0])

추출 건수: 20
첫 권: {'title': 'A Light in the Attic', 'price': '£51.77', 'rating': 3, 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


## 3. 여러 페이지 + 정제 — 중복 제거·Item 매핑

여러 페이지를 **예의를 지키며**(페이지 사이 대기) 수집하고, 중복을 제거해 Day2의 데이터
모델(`Item`)로 매핑한다. 여기서는 `url`을 자연 키로 중복을 거른다 — Day2 ERD의
`url UNIQUE`가 최종 방어선이라면, 크롤 단계에서 미리 거르는 것(교안 1.6).

가격은 `'£51.77'` 이라는 **문자열**이라 그대로는 DB에 못 넣는다 → `51.77` 로 정규화한다.

In [3]:
import re
from urllib.parse import urljoin

def crawl(pages=3, delay=0.5):
    """여러 페이지 수집 → 정제(정규화·중복 제거) → Item 리스트."""
    seen, items = set(), []
    for p in range(1, pages + 1):
        url = BASE if p == 1 else urljoin(BASE, f"catalogue/page-{p}.html")
        resp = httpx.get(url, headers=HEADERS, timeout=10)
        resp.encoding = "utf-8"
        for b in parse_books(resp.text, url):
            if b["url"] in seen:                 # 자연 키(중복 판정)
                continue
            seen.add(b["url"])
            m = re.search(r"\d+(?:\.\d+)?", b["price"])   # '£51.77' → 51.77
            items.append({
                "source": "books.toscrape.com",
                "url":    b["url"],
                "title":  b["title"],
                "price":  float(m.group()) if m else None,
                "rating": b["rating"],
            })
        time.sleep(delay)                        # 예의: 페이지 사이 대기
    return items

items = crawl()
print(f"정제 후 {len(items)}건 (중복 제거됨)")
print("샘플 Item:", items[0])

정제 후 60건 (중복 제거됨)
샘플 Item: {'source': 'books.toscrape.com', 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html', 'title': 'A Light in the Attic', 'price': 51.77, 'rating': 3}


## 4. 동적 페이지 — 화면 말고 데이터를 노려라

`books.toscrape.com` 은 정적이라 위 방식으로 전부 된다. 하지만 실무 사이트는 대개 JS가
화면을 그린다. 같은 계열의 **JS 버전**(`quotes.toscrape.com/js/`)에 같은 방식을 써 보면
**0건**이 나온다 — 그리고 데이터는 사라진 게 아니라 `<script>` 안으로 자리를 옮겼을 뿐이다(교안 1.7).

In [4]:
js_html = httpx.get("https://quotes.toscrape.com/js/",
                    headers={"User-Agent": "vibe-crawler/0.1"}, timeout=15).text
js_soup = BeautifulSoup(js_html, "html.parser")
print("정적 파싱 div.quote:", len(js_soup.select("div.quote")))

정적 파싱 div.quote: 0


In [5]:
import json as _json

m = re.search(r"var data = (\[.*?\]);", js_html, re.S)   # <script> 안 JSON 덩어리
data = _json.loads(m.group(1))
print("script 데이터에서:", len(data), "건")
print(data[0]["author"]["name"], "—", data[0]["text"][:30], "...")

script 데이터에서: 10 건
Albert Einstein — “The world as we have created  ...


In [6]:
# 무한 스크롤(/scroll)은 var data 조차 없다 — F12 Network 탭에서 API 를 찾는다
api = httpx.get("https://quotes.toscrape.com/api/quotes",
                params={"page": 1}, headers=HEADERS, timeout=15).json()
print(len(api["quotes"]), "건 | has_next =", api["has_next"])

10 건 | has_next = True


## 5. (선택) LLM 한 줄 요약 — Summarizer 예고

수집한 책들을 LLM으로 한 줄 요약한다 — 관통 프로젝트 Summarizer 부품의 축소판.
(MLAPI 키 없으면 건너뜀. 이 데이터를 요약할 때가 바로 **간접 인젝션**(2교시)이 열리는 지점 —
지금은 신뢰 가능한 데이터지만, 진짜 웹은 그렇지 않다.)

In [7]:
import os, pathlib
try:
    from dotenv import load_dotenv
    load_dotenv(pathlib.Path().resolve().parents[1] / ".env")   # 루트 .env
except Exception:
    pass

if os.getenv("MLAPI_API_KEY"):
    from openai import OpenAI
    client = OpenAI(base_url=os.getenv("MLAPI_BASE_URL"), api_key=os.getenv("MLAPI_API_KEY"))
    lines = [f"- {it['title']} (£{it['price']}, 별점 {it['rating']})" for it in items[:5]]
    prompt = "다음 책 목록을 한국어 한 문장으로 요약해줘:" + chr(10) + chr(10).join(lines)
    res = client.chat.completions.create(
        model=os.getenv("MLAPI_MODEL", "gpt-4o-mini"),
        messages=[{"role": "user", "content": prompt}],
    )
    print(res.choices[0].message.content)
else:
    print("MLAPI_API_KEY 없음 — 이 셀은 건너뜁니다(구조만 확인).")

MLAPI_API_KEY 없음 — 이 셀은 건너뜁니다(구조만 확인).


## 실습 정리

- **요청·파싱·정제**를 직접 — 에이전트가 "긁어줘"로 해 주는 일의 밑바닥.
- **예의**(User-Agent·대기·robots)는 코드에 명시했다 — 책임은 사람에게(교안 1.2 · 1.9).
- 파싱의 함정 두 개(**잘린 제목 / 클래스 이름 속 별점**)를 손으로 겪었다 — 교안 1.4의 대조 검증이 이것이다.
- 정제된 Item 이 Day2 ERD의 행이 된다 — 문서(설계)→코드(크롤러)→실물(DB)의 연결.
- 4절에서 이 데이터를 LLM에 넣었다 — **외부 데이터를 요약하는 순간이 2교시(보안)의 무대**다.